[colab에서 실행하기](https://colab.research.google.com/github/science-odysseia/AILearning/blob/main/AI_15.ipynb)

코랩에서 실행할 경우

아래 코드를 한 번 실행만 시켜 주시고 다음으로 넘어가 주세요.

In [ ]:
import os, cv2, matplotlib.pyplot as plt

if "AILearning" not in os.getcwd():
    !git clone https://github.com/science-odysseia/AILearning.git
    os.chdir("/content/AILearning")

## 생성형 AI

#### AI는 **새로운 이미지** 를 만들어낼 수 있는가?

- 기존 AI : 이미지를 보고 판단하는 것에 집중.
- 생성형 AI : 새로운 이미지를 창조. 

### Latent Space(가능성 공간)

데이터의 핵심 특징을 압축해서 표현한 숨겨진 공간

가령 어떤 28 * 28 입력 이미지를 신경망을 통과시켜 z = [0.2, -1.4, 0.8, 2.1] 이런 형식의 최종 출력을 얻었다고 하면

이 벡터가 Latent Vector가 되고, 이런 벡터들로 구성된 공간을 Latent Space 라고 한다.

### 1. GAN(Generated Adversarial Network)


- 생성자(Generator) : 가짜 이미지 생성
- 판별자(Discriminator) : 진짜/가짜 판별

![GAN_vs_Discriminator](imgs/CV0207/GAN_vs_Discriminator.png)

$$\min_{G} \max_{D} V(D, G) = \mathbb{E}_{x \sim p_{data}(x)} [\log D(x)] + \mathbb{E}_{z \sim p_{z}(z)} [\log(1 - D(G(z)))]$$

| 항목      | 의미        |
| ------- | --------- |
| D(x)    | 진짜 이미지를 True로 판별할 확률|
| D(G(z)) | 가짜 이미지를 True로 판별할 확률 |


$\mathbb{E}$ : 평균. 배치 이미지 여러 개의 Loss 평균으로 생각하면 된다.

D는 V를 최대화하려 하고, G는 V를 최소화하려 한다.

### GAN vs DCGAN

GAN : MLP. 즉 이미지를 1차원에서 처리.

DCGAN : 입력 이미지를 CNN처럼 2차원에서 처리.

### GAN의 문제점

1. 학습 불안정 : G가 좋아지면 D가 틀리고, D가 좋아지면 G가 틀리는 구조.
2. 모드 붕괴(Mode Collapse) : G는 특정 이미지가 D를 잘 속이는 것을 발견하면 해당 이미지들만 반복 생성한다. 즉 다양성이 사라진다. 

### GAN의 발전사

GAN
→ DCGAN
→ Conditional GAN
→ Pix2Pix / CycleGAN
→ WGAN
→ StyleGAN

---
### VAE(Variational AutoEncoder)

이미지를 잠재공간의 확률분포로 변환하고, 그 분포에서 샘플링해 이미지를 복원/생성하는 모델

![VAE](imgs/CV0207/VAE1.png)

VAE가 있기 전 AE(AutoEncoder)가 있었음.

![AE](imgs/CV0207/AE1.png)

AE의 구조를 예시로 MNIST 이미지 입력으로 보여주면

784차원 이미지 → Encoder → 20차원 latent vector z → Decoder → 784차원 복원 이미지

정도가 된다.

Encoder : 784 → 400 → 20 처럼 더 작은 벡터로 압축하는 신경망

Decoder : 20 → 400 → 784 처럼 latent vector z를 다시 원래 크기로 복원시키는 신경망

AE의 목표는 입력 x와 출력 x'가 최대한 비슷해지게 만드는 것이다.


---

AE는 Encoder에서 z를 생성했다면

VAE는 Encoder에서 z 대신 $\mu$, $\sigma$ 를 생성한다.

그리고 이 $\mu$, $\sigma$ 를 이용해 z를 생성하는데,

이 과정을 **재매개변수화(reparameterization)** 이라고 한다.

$z = \mu + \sigma \cdot \epsilon  (\epsilon \sim N(0, 1))$

가령 예를 들어서 Encoder : 784 → 400 → 20 구조를 예시로 들어보면

일단 `self.fc = nn.Linear(784, 400)`로 신경망을 한번 통과시키고

이후 `self.mu = nn.Linear(400, 20)`, `self.var = nn.Linear(400, 20)`로 

평균과 분산(정확히는 분산의 로그값)으로 쓸 값들을 생성한다.

이 때, 분산은 $\sigma^2$ 으로 항상 양수가 나와야 하나 출력층의 값들은 음수가 나올 수도 있기 때문에

log(σ²)를 통해 스케일차이를 안정화 시켜 준 다음(값이 너무 커지거나 작은 경우를 대비한 분포를 줄여주는 스킬 정도로 생각하면 됨)

$\sigma = e ^ {\frac{1}{2} \log {\sigma ^ 2}}$ 공식을 적용. (코드는 `std = torch.exp(0.5 * var)`)

$z = \mu + \sigma \cdot \epsilon  (\epsilon \sim N(0, 1))$

에서 사용할 랜덤 노이즈 $\epsilon$ 을 생성. (코드는 `eps = torch.randn_like(std)`)

이런 식으로 잠재벡터(Latent Vector) z를 생성한다.

---
### Conditional GAN

기존 GAN은 사용자가 원하는 이미지만을 생성하기 어렵다.

예를 들어 MNIST 이미지를 입력으로 넣었을 때

"7 이미지만 만들어줘"라고 명령하기 어렵다는 의미이다.

Conditional GAN은 Generator에 조건 y를 같이 넣는다.

    z + y → Generator → fake image

가령 MNIST를 예로 들면

숫자 7 이미지로만 설정하고싶다면

원핫벡터 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0]을 

랜덤 노이즈 입력 벡터 z와 같이 넣는다.

가량 노이즈 입력 벡터 z를 100차원으로 설정하면

원핫벡터와 합친 110차원의 입력 벡터 `concat(z, y)`를 넣는 것이다.

Discriminator도 이미지와 y값을 같이 받는다.

이유는 입력 이미지가 진짜인지 가짜인지 여부도 판단해야 하지만

조건 y에 부합하는지도 판단해야 하기 때문이다.

예를 들어 y = 7인 상황에서 3을 그린 이미지를 받았다면

이는 조건에 맞지 않는 이미지가 된다.

---
### pix2pix

cGAN이 입력 노이즈 z벡터와 라벨 y를 같이 넣었다면

pix2pix에서는 라벨 y 대신

아래 이미지처럼 형태만 갖춘 형태이미지를 직접 생성기에 넣는 방식이다.

![pix2pix](imgs/CV0207/pix2pix1.png)

---
### U-net

![Unet](imgs/CV0207/Unet.png)

pix2pix에서 Generator에서 사용되는 것이 바로 이 U-net이다.

CNN이 단순 클래스 분류뿐 아니라 픽셀 단위 출력(segmentation 등)을 요구하는 문제에도 사용되기 시작하였고,

이에 따라 입력 이미지를 압축하여 특징을 추출한 뒤, 다시 원본 크기로 복원하는 Encoder-Decoder 구조가 등장하였다.

하지만 Encoder-Decoder는 압축 과정에서 해상도가 크게 감소하면서, 경계선이나 위치 같은 세부 정보를 잃어버리는 문제가 있었는데.

이를 해결하기 위해 U-Net은 Encoder의 feature map을 Decoder로 직접 전달하는 Skip Connection 구조를 도입하였다.

### U-Net의 기본 블록: DoubleConv

U-Net의 가장 기본이 되는 블록

### 구조
```
입력 → Conv 3x3 → ReLU → Conv 3x3 → ReLU → 출력
```

### 특징
- 3x3 Convolution을 **2번 연속** 수행
- 각 Conv 후 ReLU 활성화 함수 적용

### 왜 2번 수행하는가?
- 더 복잡한 특징을 학습할 수 있음
- 비선형성 증가 (ReLU 2번)

### Pix2Pix 구조

#### G
생성자의 구조는 U-net과 동일

입력 : 형태 이미지 x

출력 : 가짜 이미지 G(x)

#### D

판별자는 주어진 형태 이미지 x와 진짜 이미지 y의 concat 형태((x, y))

또는 주어진 형태 이미지 x와 생성자가 만든 가짜 이미지 G(x)의 concat 형태((x, G(x)))

이 둘 중 하나의 형태를 입력으로 받는다.

가령 두 이미지 모두 128 x 128 x 3의 형태라고 가정하면

두 이미지를 채널 방향으로 합친 (Batch, 6, 128, 128)형태로 입력을 받는다는 의미이다.